In [ ]:
# agent 코드에 수합될 task 4코드를 작성해줘
# task 4코드는 Agent.ipynb파일에 포함될 node중하나로 구성될 것이고 아래 평가기준에 맞게끔 재질의(멀티턴)로직이 구현된 FunctionCalling구조야.
# Example ) '최근 많이 오른 지식이 뭐야?'라는 질문이 들어오면, '최근을 정의 하기위한 정확한 날짜를 말해주세요' 라는 답변이 나오고 최종적으로 

# 과제에서 제시한 조건 : 야후파이낸스 API사용 해서 아래 과제 5개에 맞게 답변
# * Task1 - 단순 종목/날짜조회
#    * {종목명}의 {날짜}종가는?
#    * {날짜}에서 거래량 기준 상위 10개 종목은?
# * Task2 - 거래량 조건 검색
#    * 거래량이 전날대비{숫자}% 이상 오른 종목을 모두 보여줘
# * Task3 - 시그널 감지
#    * {날짜} 기준 50일 이동평균을 10%이상 상향 돌파한 종목을 알려줘
# * Task4 - 모호한 의미해석 (ex. 최근 많이 오른 주식, 고점 대비 떨어진 주식 등)
#    * 최근 많이 오른 주식을 알려줘
#    * 고점 대비 많이 떨어진 주식을 알려줘
#    * 평가기준 :
#       * 사람의 모호한 언어를 정량화 하여 표현하는 로직이 고려되었는가
#       * 되묻기(멀티턴) 또는 재질의(재질문) 요구 등 신뢰성있는 답변을 하기 위한 노력이 추가 되었는가

In [ ]:
# 실행해보기
# 아래 셀을 실행하면 KOSPI 데이터와 한국 주식 데이터를 확인할 수 있습니다.

In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
# KRX 티커 파일을 사용한 대규모 다운로드 속도 비교

import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import threading

def load_krx_tickers(market="ALL", sample_size=50):
    """KRX 티커 리스트 로드"""
    try:
        tickers_df = pd.read_csv(
            "/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/app/data/krx_tickers_with_market.csv",
            dtype={"code": str}
        )
    except FileNotFoundError:
        try:
            tickers_df = pd.read_csv(
                "/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/sasha/test/krx_tickers_cleaned.csv",
                dtype={"code": str}
            )
        except FileNotFoundError:
            print("❌ 종목 리스트 파일을 찾을 수 없습니다.")
            return []
    
    if market.upper() != "ALL":
        tickers_df = tickers_df[tickers_df["market"].str.upper() == market.upper()]
    
    # 샘플링
    if len(tickers_df) > sample_size:
        tickers_df = tickers_df.sample(n=sample_size, random_state=42)
    
    tickers_df["yf_ticker"] = tickers_df.apply(
        lambda x: f"{x['code']}.KS" if x['market'].upper() == "KOSPI" else f"{x['code']}.KQ", 
        axis=1
    )
    
    return tickers_df["yf_ticker"].tolist()

def method1_sequential(tickers, period="1mo"):
    """방법 1: 순차적 다운로드"""
    start_time = time.time()
    success_count = 0
    
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            if not data.empty:
                success_count += 1
        except:
            continue
    
    return time.time() - start_time, success_count

def method2_batch_small(tickers, period="1mo", batch_size=10):
    """방법 2: 소규모 배치 다운로드"""
    start_time = time.time()
    success_count = 0
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            data = yf.download(
                batch,
                period=period,
                group_by="ticker",
                progress=False,
                auto_adjust=True,
                threads=False
            )
            if not data.empty:
                success_count += len(batch)
        except:
            continue
    
    return time.time() - start_time, success_count

def method3_batch_large(tickers, period="1mo"):
    """방법 3: 대규모 배치 다운로드"""
    start_time = time.time()
    
    try:
        data = yf.download(
            tickers,
            period=period,
            group_by="ticker",
            progress=False,
            auto_adjust=True,
            threads=False
        )
        success_count = len(tickers) if not data.empty else 0
    except:
        success_count = 0
    
    return time.time() - start_time, success_count

def method4_threading(tickers, period="1mo", max_workers=10):
    """방법 4: 멀티스레딩"""
    start_time = time.time()
    success_count = 0
    
    def fetch_single(ticker):
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            return not data.empty
        except:
            return False
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(fetch_single, tickers))
        success_count = sum(results)
    
    return time.time() - start_time, success_count

# 테스트 실행
print("=== KRX 티커를 이용한 대규모 다운로드 속도 비교 ===\n")

# 티커 로드
sample_sizes = [10, 30, 50]

for size in sample_sizes:
    print(f"📊 {size}개 종목 테스트")
    print("-" * 40)
    
    tickers = load_krx_tickers(market="ALL", sample_size=size)
    
    if not tickers:
        print("❌ 티커를 로드할 수 없습니다.\n")
        continue
    
    print(f"로드된 티커 수: {len(tickers)}")
    
    # 방법 1: 순차적
    time1, success1 = method1_sequential(tickers[:10] if size > 10 else tickers)  # 시간 절약을 위해 제한
    print(f"1. 순차적 다운로드: {time1:.2f}초 (성공: {success1}개)")
    
    # 방법 2: 소규모 배치
    time2, success2 = method2_batch_small(tickers, batch_size=5)
    print(f"2. 소규모 배치: {time2:.2f}초 (성공: {success2}개)")
    
    # 방법 3: 대규모 배치
    time3, success3 = method3_batch_large(tickers)
    print(f"3. 대규모 배치: {time3:.2f}초 (성공: {success3}개)")
    
    # 방법 4: 멀티스레딩 (작은 샘플만)
    if size <= 20:
        time4, success4 = method4_threading(tickers, max_workers=5)
        print(f"4. 멀티스레딩: {time4:.2f}초 (성공: {success4}개)")
    
    # 가장 빠른 방법 찾기
    times = [time2, time3]
    methods = ["소규모 배치", "대규모 배치"]
    if size <= 20:
        times.append(time4)
        methods.append("멀티스레딩")
    
    fastest_idx = times.index(min(times))
    print(f"🏆 가장 빠른 방법: {methods[fastest_idx]} ({min(times):.2f}초)")
    print()

print("=== 권장사항 ===")
print("• 10개 이하: 순차적 다운로드")
print("• 10-50개: 소규모 배치 (5-10개씩)")
print("• 50개 이상: 대규모 배치 (주의: 실패 위험)")
print("• 안정성 중시: 멀티스레딩 (max_workers=5-10)")

In [3]:
# KOSPI 지수 심볼: ^KS11
# 개별 한국 주식 심볼 예시: 005930.KS (삼성전자), 000660.KS (SK하이닉스)

period = "1y"  # 데이터 기간 설정 (예: "1d", "5d", "1mo", "3mo", "6mo", "1y", "2y", "5y", "10y", "ytd", "max")
symbol = "005930.KS"  # 삼성전자 예시

kospi = yf.Ticker(symbol)
data = kospi.history(period=period)

data

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-07-29 00:00:00+09:00,80119.155018,80511.895974,79628.228824,79726.414062,12797136,0.0,0.0
2024-07-30 00:00:00+09:00,78940.935417,79530.046875,78548.194444,79530.046875,13169636,0.0,0.0
2024-07-31 00:00:00+09:00,79726.412656,82377.414062,79431.856945,82377.414062,20744323,0.0,0.0
2024-08-01 00:00:00+09:00,84439.301482,84537.486716,81591.929688,81591.929688,20900338,0.0,0.0
2024-08-02 00:00:00+09:00,79530.038572,79922.779503,78057.260080,78155.445312,25800275,0.0,0.0
...,...,...,...,...,...,...,...
2025-07-22 00:00:00+09:00,68100.000000,68500.000000,65600.000000,66000.000000,20829006,0.0,0.0
2025-07-23 00:00:00+09:00,66200.000000,66500.000000,64900.000000,66400.000000,15687058,0.0,0.0
2025-07-24 00:00:00+09:00,66500.000000,66800.000000,65900.000000,66000.000000,12660193,0.0,0.0


In [7]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import asyncio
import concurrent.futures
from typing import List, Tuple, Optional
import threading
import queue

# 방법 1: 스트리밍 방식 - 실시간 결과 출력
def stream_top_performers(tickers: List[str], period_days: int = 30, top_n: int = 10):
    """
    실시간으로 결과를 스트리밍하며 상위 종목 찾기
    """
    results = []
    processed_count = 0
    total_count = len(tickers)
    
    print(f"총 {total_count}개 종목 분석 시작...")
    
    for ticker in tickers:
        try:
            # 개별 종목 빠른 다운로드
            stock = yf.Ticker(ticker)
            hist = stock.history(period=f"{period_days}d", auto_adjust=True)
            
            if len(hist) >= 2:
                start_price = hist['Close'].iloc[0]
                end_price = hist['Close'].iloc[-1]
                return_pct = ((end_price / start_price) - 1) * 100
                
                results.append((ticker, return_pct))
                
                # 실시간 상위 10개 업데이트
                results.sort(key=lambda x: x[1], reverse=True)
                if len(results) > top_n:
                    results = results[:top_n]
                
                processed_count += 1
                
                # 진행상황과 현재 상위 종목 출력
                if processed_count % 50 == 0 or processed_count <= 10:
                    print(f"\n진행: {processed_count}/{total_count}")
                    print("현재 상위 5개:")
                    for i, (sym, ret) in enumerate(results[:5], 1):
                        print(f"  {i}. {sym}: {ret:.2f}%")
            
        except Exception as e:
            continue
    
    return results[:top_n]

# 방법 2: 멀티스레딩 - 병렬 처리
def parallel_top_performers(tickers: List[str], period_days: int = 30, top_n: int = 10, max_workers: int = 20):
    """
    멀티스레딩으로 병렬 처리
    """
    def fetch_single_return(ticker):
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period=f"{period_days}d", auto_adjust=True)
            
            if len(hist) >= 2:
                return_pct = ((hist['Close'].iloc[-1] / hist['Close'].iloc[0]) - 1) * 100
                return (ticker, return_pct)
        except:
            pass
        return None
    
    results = []
    completed = 0
    total = len(tickers)
    
    print(f"병렬 처리 시작 (워커: {max_workers}개)...")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 모든 작업 제출
        future_to_ticker = {executor.submit(fetch_single_return, ticker): ticker for ticker in tickers}
        
        # 완료되는 대로 결과 수집
        for future in concurrent.futures.as_completed(future_to_ticker):
            result = future.result()
            if result:
                results.append(result)
            
            completed += 1
            if completed % 100 == 0:
                print(f"진행: {completed}/{total}")
    
    # 상위 종목 정렬
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# 방법 3: 적응형 배치 - 작은 배치로 점진적 처리
def adaptive_batch_processing(tickers: List[str], period_days: int = 30, top_n: int = 10):
    """
    작은 배치로 나누어 점진적 처리
    """
    batch_size = 20  # 작은 배치 크기
    all_results = []
    
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i + batch_size]
        print(f"배치 {i//batch_size + 1}/{(len(tickers)-1)//batch_size + 1} 처리 중...")
        
        try:
            # 작은 배치 다운로드 (빠름)
            end_date = datetime.now()
            start_date = end_date - timedelta(days=period_days + 10)
            
            data = yf.download(
                batch,
                start=start_date,
                end=end_date,
                group_by="ticker",
                progress=False,
                auto_adjust=True,
                threads=False
            )
            
            # 배치 결과 처리
            batch_results = []
            for ticker in batch:
                try:
                    if len(batch) == 1:
                        ticker_data = data
                    else:
                        ticker_data = data[ticker] if ticker in data.columns.get_level_values(0) else None
                    
                    if ticker_data is not None and 'Close' in ticker_data.columns:
                        close_prices = ticker_data['Close'].dropna()
                        if len(close_prices) >= 2:
                            return_pct = ((close_prices.iloc[-1] / close_prices.iloc[0]) - 1) * 100
                            batch_results.append((ticker, return_pct))
                
                except Exception:
                    continue
            
            all_results.extend(batch_results)
            
            # 중간 결과 출력
            all_results.sort(key=lambda x: x[1], reverse=True)
            print(f"현재 상위 3개: {[(t, f'{r:.2f}%') for t, r in all_results[:3]]}")
            
        except Exception as e:
            print(f"배치 {i//batch_size + 1} 처리 실패: {e}")
            continue
    
    return all_results[:top_n]

# 방법 4: 캐시 활용 - 이전 결과 재사용
class CachedAnalyzer:
    def __init__(self):
        self.cache = {}
        self.cache_expiry = {}
    
    def get_top_performers_cached(self, tickers: List[str], period_days: int = 30, top_n: int = 10):
        """캐시를 활용한 분석"""
        current_time = datetime.now()
        cache_duration = timedelta(hours=1)  # 1시간 캐시
        
        fresh_results = []
        tickers_to_fetch = []
        
        # 캐시 확인
        for ticker in tickers:
            cache_key = f"{ticker}_{period_days}"
            if (cache_key in self.cache and 
                cache_key in self.cache_expiry and 
                current_time < self.cache_expiry[cache_key]):
                fresh_results.append(self.cache[cache_key])
            else:
                tickers_to_fetch.append(ticker)
        
        print(f"캐시에서 {len(fresh_results)}개, 새로 가져올 {len(tickers_to_fetch)}개")
        
        # 새 데이터 가져오기 (병렬 처리)
        if tickers_to_fetch:
            new_results = parallel_top_performers(tickers_to_fetch, period_days, len(tickers_to_fetch))
            
            # 캐시 업데이트
            for ticker, return_pct in new_results:
                cache_key = f"{ticker}_{period_days}"
                self.cache[cache_key] = (ticker, return_pct)
                self.cache_expiry[cache_key] = current_time + cache_duration
            
            fresh_results.extend(new_results)
        
        # 최종 정렬
        fresh_results.sort(key=lambda x: x[1], reverse=True)
        return fresh_results[:top_n]

# 사용 예시
if __name__ == "__main__":
    # 테스트용 종목 리스트 (실제로는 더 많은 종목)
    test_tickers = ["AAPL", "GOOGL", "MSFT", "TSLA", "AMZN", "META", "NVDA", "NFLX", "CRM", "ADBE"]
    
    print("=== 방법 1: 스트리밍 방식 ===")
    stream_results = stream_top_performers(test_tickers, 30, 5)
    print("\n=== 방법 1 결과 ===")
    for ticker, return_pct in stream_results:
        print(f"{ticker}: {return_pct:.2f}%")
        
    
    print("\n=== 방법 2: 병렬 처리 ===")
    parallel_results = parallel_top_performers(test_tickers, 30, 5)
    print("\n=== 방법 2 결과 ===")
    for ticker, return_pct in parallel_results:
        print(f"{ticker}: {return_pct:.2f}%")
    
    print("\n=== 방법 3: 적응형 배치 ===")
    adaptive_results = adaptive_batch_processing(test_tickers, 30, 5)
    print("\n=== 방법 3 결과 ===")
    for ticker, return_pct in adaptive_results:
        print(f"{ticker}: {return_pct:.2f}%")
    
    print("\n=== 방법 4: 캐시 활용 ===")
    analyzer = CachedAnalyzer()
    cached_results = analyzer.get_top_performers_cached(test_tickers, 30, 5)
    print("\n=== 방법 4 결과 ===")
    for ticker, return_pct in cached_results:
        print(f"{ticker}: {return_pct:.2f}%")

=== 방법 1: 스트리밍 방식 ===
총 10개 종목 분석 시작...

진행: 1/10
현재 상위 5개:
  1. AAPL: 7.37%

진행: 2/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AAPL: 7.37%

진행: 3/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AAPL: 7.37%
  3. MSFT: 7.28%

진행: 4/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AAPL: 7.37%
  3. MSFT: 7.28%
  4. TSLA: -0.96%

진행: 5/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AMZN: 8.53%
  3. AAPL: 7.37%
  4. MSFT: 7.28%
  5. TSLA: -0.96%

진행: 6/10
현재 상위 5개:
  1. GOOGL: 9.95%
  2. AMZN: 8.53%
  3. AAPL: 7.37%
  4. MSFT: 7.28%
  5. META: 2.87%

진행: 7/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

진행: 8/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

진행: 9/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

진행: 10/10
현재 상위 5개:
  1. NVDA: 19.66%
  2. GOOGL: 9.95%
  3. AMZN: 8.53%
  4. AAPL: 7.37%
  5. MSFT: 7.28%

=== 방법 1 결과 ===
NVDA: 19.66%
GOOGL: 9.95%
AMZN: 8.53%
AAPL: 7.3

In [8]:
# 각 방법별 시간 측정 및 비교 테스트

import time
from datetime import datetime, timedelta

def time_single_download():
    """개별 다운로드 방식 시간 측정"""
    tickers = ["005930.KS", "000660.KS", "035420.KS","051910.KS", "006400.KS"]

    start_time = time.time()
    results = []

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period="1mo")
            results.append(data)
        except:
            continue

    end_time = time.time()
    return end_time - start_time, len(results)

def time_batch_download():
    """배치 다운로드 방식 시간 측정"""
    tickers = ["005930.KS", "000660.KS", "035420.KS","051910.KS", "006400.KS"]

    start_time = time.time()

    try:
        data = yf.download(
            tickers,
            period="1mo",
            group_by="ticker",
            progress=False,
            auto_adjust=True,
            threads=False
        )
        success = not data.empty
    except:
        success = False

    end_time = time.time()
    return end_time - start_time, success

def time_kospi_download():
    """KOSPI 지수 다운로드 시간 측정"""
    start_time = time.time()

    try:
        kospi = yf.Ticker("^KS11")
        data = kospi.history(period="1mo")
        success = not data.empty
    except:
        success = False

    end_time = time.time()
    return end_time - start_time, success

# 시간 측정 실행
print("=== yfinance 다운로드 방식별 시간 비교 ===\n")

# 1. 개별 다운로드
print("1. 개별 다운로드 방식 (5개 종목)")
single_time, single_count = time_single_download()
print(f"   소요시간: {single_time:.2f}초")
print(f"   성공한 종목: {single_count}개")

print()

# 2. 배치 다운로드
print("2. 배치 다운로드 방식 (5개 종목)")
batch_time, batch_success = time_batch_download()
print(f"   소요시간: {batch_time:.2f}초")
print(f"   성공 여부: {'성공' if batch_success else '실패'}")

print()

# 3. KOSPI 지수
print("3. KOSPI 지수 다운로드")
kospi_time, kospi_success = time_kospi_download()
print(f"   소요시간: {kospi_time:.2f}초")
print(f"   성공 여부: {'성공' if kospi_success else '실패'}")

print()

# 결과 비교
print("=== 결과 분석 ===")
if single_count > 0:
    avg_single = single_time / single_count
    print(f"개별 다운로드 - 종목당 평균: {avg_single:.2f}초")

if batch_success and single_count > 0:
    efficiency = single_time / batch_time if batch_time > 0 else 0
    print(f"배치 다운로드 효율성: {efficiency:.1f}배 {'빠름' if efficiency > 1 else '느림'}")

print(f"KOSPI 지수 다운로드가 가장 {'빠름' if kospi_time <= min(single_time, batch_time) else '느림'}")

=== yfinance 다운로드 방식별 시간 비교 ===

1. 개별 다운로드 방식 (5개 종목)
   소요시간: 1.03초
   성공한 종목: 5개

2. 배치 다운로드 방식 (5개 종목)
   소요시간: 1.33초
   성공 여부: 성공

3. KOSPI 지수 다운로드
   소요시간: 0.36초
   성공 여부: 성공

=== 결과 분석 ===
개별 다운로드 - 종목당 평균: 0.21초
배치 다운로드 효율성: 0.8배 느림
KOSPI 지수 다운로드가 가장 빠름


In [9]:
# KRX 티커 파일을 사용한 대규모 다운로드 속도 비교

import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import threading

def load_krx_tickers(market="ALL", sample_size=50):
    """KRX 티커 리스트 로드"""
    try:
        tickers_df = pd.read_csv(
            "/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/app/data/krx_tickers_with_market.csv",
            dtype={"code": str}
        )
    except FileNotFoundError:
        try:
            tickers_df = pd.read_csv("/mnt/c/Users/rladl/Desktop/Mirae_Agent/MIRAE_ASSET_AI-Festival/sasha/test/krx_tickers_cleaned.csv",
                dtype={"code": str}
            )
        except FileNotFoundError:
            print("❌ 종목 리스트 파일을 찾을 수 없습니다.")
            return []

    if market.upper() != "ALL":
        tickers_df = tickers_df[tickers_df["market"].str.upper() == market.upper()]

    # 샘플링
    if len(tickers_df) > sample_size:
        tickers_df = tickers_df.sample(n=sample_size,random_state=42)

    tickers_df["yf_ticker"] = tickers_df.apply(
        lambda x: f"{x['code']}.KS" if x['market'].upper() == "KOSPI" else f"{x['code']}.KQ",axis=1
    )

    return tickers_df["yf_ticker"].tolist()

def method1_sequential(tickers, period="1mo"):
    """방법 1: 순차적 다운로드"""
    start_time = time.time()
    success_count = 0

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            if not data.empty:
                success_count += 1
        except:
            continue

    return time.time() - start_time, success_count

def method2_batch_small(tickers, period="1mo", batch_size=10):
    """방법 2: 소규모 배치 다운로드"""
    start_time = time.time()
    success_count = 0

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            data = yf.download(
                batch,
                period=period,
                group_by="ticker",
                progress=False,
                auto_adjust=True,
                threads=False
            )
            if not data.empty:
                success_count += len(batch)
        except:
            continue

    return time.time() - start_time, success_count

def method3_batch_large(tickers, period="1mo"):
    """방법 3: 대규모 배치 다운로드"""
    start_time = time.time()

    try:
        data = yf.download(
            tickers,
            period=period,
            group_by="ticker",
            progress=False,
            auto_adjust=True,
            threads=False
        )
        success_count = len(tickers) if not data.empty else 0
    except:
        success_count = 0

    return time.time() - start_time, success_count

def method4_threading(tickers, period="1mo", max_workers=10):
    """방법 4: 멀티스레딩"""
    start_time = time.time()
    success_count = 0

    def fetch_single(ticker):
        try:
            stock = yf.Ticker(ticker)
            data = stock.history(period=period)
            return not data.empty
        except:
            return False

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(fetch_single,tickers))
        success_count = sum(results)

    return time.time() - start_time, success_count


## Dart

In [1]:
!pip install dart-fss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 2.8 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [dart-fss]/14 [dart-fss]lease]


In [2]:
import dart_fss as dart_fss
import pandas as pd

api_key = '	0778fdc977a04c6bc0516326fe22cea0224b233b' #'여기에 API 키를 입력'
dart_fss.set_api_key(api_key=api_key)

corp_list = dart_fss.get_corp_list()

corp_list.corps

Error occurred during getting browser(s): random, but was suppressed with fallback.


[[00434003]다코,
 [00430964]굿앤엘에스,
 [00388953]크레디피아제이십오차유동화전문회사,
 [00179984]연방건설산업,
 [00420143]브룩스피알아이오토메이션잉크,
 [00401111]매경아이비아이,
 [00435534]캐드뱅크,
 [00430186]엠와이오피삼차유동화전문유한회사,
 [00430201]엠와이오피이차유동화전문유한회사,
 [00430210]엠와이오피일차유동화전문유한회사,
 [00430229]포스미디어,
 [00140432]축복할렐루야,
 [00426208]한국전자화학,
 [00433262]ConnachtCapitalMarketInvestmentLtd.,
 [00433749]선진아이티,
 [00433785]팀스코리아,
 [00196079]에넥스하이테크,
 [00435048]세이스텝바이스텝혼합형펀드,
 [00435057]유리알파헷지채권혼합형펀드,
 [00108843]한기술정보통신,
 [00443232]다움종합건설,
 [00248293]한국애치슨,
 [00455662]고은상사,
 [00149318]조흥종합건설,
 [00451347]유니즈유통,
 [00415105]창대화장품,
 [00420824]문화종합건설,
 [00389439]한국종합미디어,
 [00454016]아이펜텍,
 [00452072]대홍염공,
 [00437602]지오항공여행사,
 [00365518]헤이아니타코리아,
 [00246967]태산산업개발,
 [00445559]군포종합시장,
 [00444657]기우,
 [00448936]라시도,
 [00405658]예전미디어,
 [00410818]세미가,
 [00416061]대닉스,
 [00448909]인포핸드,
 [00445407]지현개발,
 [00421647]한국부동산신탁제일차유동화전문유한회사,
 [00421425]ISCCaymanLtd.,
 [00423265]연산,
 [00423292]씨케이알유동화전문유한회사,
 [00420602]디에스피,
 [00307505]LOMBARDKOREAILIMITED,
 [00241397

## 기업코드 매핑

In [24]:
import dart_fss
import pandas as pd

# DART 인증키 설정
dart_fss.set_api_key('0778fdc977a04c6bc0516326fe22cea0224b233b')

# 상장법인 목록 불러오기
corp_list = dart_fss.api.filings.get_corp_code()

# 바로 DataFrame으로 변환
corp_df = pd.DataFrame(corp_list)

# 필요한 컬럼 추출
corp_df = corp_df[['corp_code', 'corp_name', 'stock_code']]

# 상장기업만 필터링 (stock_code 없는 기업 제외)
corp_df = corp_df[corp_df['stock_code'].notnull() & (corp_df['stock_code'] != '')]


In [25]:
# 기존 KRX 데이터 로딩
krx_df = pd.read_csv('krx_tickers_cleaned.csv', dtype={'code': str})

# code → 6자리로 맞추기
krx_df['code'] = krx_df['code'].str.zfill(6)

# 병합: stock_code(code) 기준
merged_df = krx_df.merge(corp_df, left_on='code', right_on='stock_code', how='left')

# 필요한 컬럼만 정리 + 컬럼명 통일
merged_df = merged_df[['code', 'name', 'market', 'corp_code']]


In [27]:
# CSV로 저장
merged_df.to_csv('krx_tickers_corp_code.csv', index=False, encoding='utf-8-sig')

In [28]:
merged_df

,code,name,market,corp_code
0,060310,3S,KOSDAQ,00378363
1,095570,AJ네트웍스,KOSPI,00365387
2,006840,AK홀딩스,KOSPI,00125080
3,054620,APS,KOSDAQ,00296078
4,265520,AP시스템,KOSDAQ,01203808
...,...,...,...,...
2752,000540,흥국화재,KOSPI,00103176
2753,000545,흥국화재우,KOSPI,NaN
2754,003280,흥아해운,KOSPI,00167208
2755,037440,희림,KOSDAQ,00236863


In [4]:
corp_code = df_listed[df_listed['corp_name'] == '삼성전자'].iloc[0,0]

corp_code

'00126380'

In [5]:
corp_code = df_listed[df_listed['corp_name'] == '카카오'].iloc[0,0]
data = dart_fss.api.finance.fnltt_singl_acnt(corp_code, '2021', '11011')

pd.DataFrame(data['list'])

,rcept_no,reprt_code,bsns_year,corp_code,stock_code,fs_div,fs_nm,sj_div,sj_nm,account_nm,...,thstrm_dt,thstrm_amount,frmtrm_nm,frmtrm_dt,frmtrm_amount,bfefrmtrm_nm,bfefrmtrm_dt,bfefrmtrm_amount,ord,currency
0,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,유동자산,...,2021.12.31 현재,"8,102,194,310,997",제 26 기,2020.12.31 현재,"4,462,924,201,049",제 25 기,2019.12.31 현재,"2,829,694,454,045",1,KRW
1,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,비유동자산,...,2021.12.31 현재,"13,716,985,880,517",제 26 기,2020.12.31 현재,"6,987,572,257,652",제 25 기,2019.12.31 현재,"5,907,674,589,688",3,KRW
2,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,자산총계,...,2021.12.31 현재,"22,784,559,108,590",제 26 기,2020.12.31 현재,"11,954,146,190,738",제 25 기,2019.12.31 현재,"8,737,369,043,733",5,KRW
3,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,유동부채,...,2021.12.31 현재,"5,657,183,444,239",제 26 기,2020.12.31 현재,"2,949,489,217,453",제 25 기,2019.12.31 현재,"2,447,884,951,785",7,KRW
4,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,비유동부채,...,2021.12.31 현재,"2,884,801,635,871",제 26 기,2020.12.31 현재,"1,162,344,689,626",제 25 기,2019.12.31 현재,"560,955,127,367",9,KRW
5,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,부채총계,...,2021.12.31 현재,"9,216,225,462,911",제 26 기,2020.12.31 현재,"4,542,386,717,365",제 25 기,2019.12.31 현재,"3,008,840,079,152",11,KRW
6,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,자본금,...,2021.12.31 현재,"44,640,599,000",제 26 기,2020.12.31 현재,"44,300,999,000",제 25 기,2019.12.31 현재,"43,161,318,000",13,KRW
7,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,이익잉여금,...,2021.12.31 현재,"1,689,444,597,928",제 26 기,2020.12.31 현재,"301,009,092,939",제 25 기,2019.12.31 현재,"136,861,072,078",17,KRW
8,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,BS,재무상태표,자본총계,...,2021.12.31 현재,"13,568,333,645,679",제 26 기,2020.12.31 현재,"7,411,759,473,373",제 25 기,2019.12.31 현재,"5,728,528,964,581",21,KRW
9,20240320000386,11011,2021,00258801,035720,CFS,연결재무제표,IS,손익계산서,영업이익,...,2021.01.01 ~ 2021.12.31,"587,884,339,202",제 26 기,2020.01.01 ~ 2020.12.31,"450,519,574,781",제 25 기,2019.01.01 ~ 2019.12.31,"202,191,335,143",25,KRW


In [22]:

import yfinance as yf
from datetime import timedelta
from typing import Dict, Any

def _get_stock_info_at_date(ticker: str, date: str) -> Dict[str, Any]:
    """특정 날짜의 주식 정보 수집 (yfinance)"""
    try:
        stock = yf.Ticker(ticker)
        target_date = pd.to_datetime(date).tz_localize(None)
        start_date = target_date - timedelta(days=30)
        end_date = target_date + timedelta(days=1)

        hist = stock.history(start=start_date, end=end_date)

        if hist.empty:
            return {"error": "해당 날짜의 주가 데이터를 찾을 수 없습니다."}

        hist.index = hist.index.tz_localize(None)  # ✅ 타임존 제거

        closest_date = hist.index[hist.index <= target_date].max()
        if pd.isna(closest_date):
            closest_date = hist.index.min()

        price_data = hist.loc[closest_date]
        info = stock.info
        # news = _get_news_around_date(stock, target_date)

        return {
            "date": closest_date.strftime("%Y-%m-%d"),
            "price": {
                "close": float(price_data["Close"]),
                "volume": int(price_data["Volume"]),
                "high": float(price_data["High"]),
                "low": float(price_data["Low"])
            },
            "valuation": {
                "market_cap": info.get("marketCap", 0),
                "pe_ratio": info.get("trailingPE"),
                "pb_ratio": info.get("priceToBook"),
                "eps": info.get("trailingEps"),
                "dividend_yield": info.get("dividendYield")
            },
            "analyst": {
                "recommendation": info.get("recommendationKey"),
                "target_price": info.get("targetMeanPrice"),
                "analyst_count": info.get("numberOfAnalystOpinions")
            },
            # "news": news[:5],
            "sector": info.get("sector"),
            "industry": info.get("industry")
        }
    except Exception as e:
        return {"error": f"데이터 수집 중 오류: {str(e)}"}


print(_get_stock_info_at_date( "인산가", "2024-12-15"))


HTTP Error 404: 
$인산가: possibly delisted; no timezone found


{'error': '해당 날짜의 주가 데이터를 찾을 수 없습니다.'}


In [6]:
# //시총 Top 50 회사 연봉왕 뽑아보기 -  카카오의 corp_code 구하기

corp_code = df_listed[df_listed['corp_name'] == '카카오'].iloc[0,0]
data = dart_fss.api.info.indvdl_by_pay(corp_code, '2021', '11011')

df_temp = pd.DataFrame(data['list'])
df_temp = df_temp[['corp_name','nm','ofcps','mendng_totamt']]
df_temp.columns = ['기업명','이름','역할','보수']

df_temp

,기업명,이름,역할,보수
0,카카오,신정환,신사업 담당,"12,874,000,000"
1,카카오,배재현,CAC 투자거버넌스 총괄,"8,261,000,000"
2,카카오,권승조,(前) 지적재산부문 책임자,"6,535,000,000"
3,카카오,남궁훈,카카오 OTF 장,"6,158,000,000"
4,카카오,안성진,(前) M사업전략 자문,"5,750,000,000"


In [9]:
# // 함수로 만들기

def get_salary_top(name):
  corp_code = df_listed[df_listed['corp_name'] == name].iloc[0,0]
  data = dart_fss.api.info.indvdl_by_pay(corp_code, '2024', '11011')
  
  df_temp = pd.DataFrame(data['list'])
  df_temp = df_temp[['corp_name','nm','ofcps','mendng_totamt']]
  df_temp.columns = ['기업명','이름','역할','보수']
  df_temp['보수'] = pd.to_numeric(df_temp['보수'].str.replace(',',''))
  df_temp = df_temp.sort_values(by='보수',ascending=False)
  
  return df_temp

In [10]:
# // 여러개 종목에 대해 뽑아보기

names = ['삼성전자','LG에너지솔루션','SK하이닉스','NAVER','삼성바이오로직스','삼성전자우','카카오','삼성SDI','현대차','LG화학','기아','POSCO홀딩스','KB금융','카카오뱅크','셀트리온','신한지주','삼성물산','현대모비스','SK이노베이션','LG전자','카카오페이','SK','한국전력','크래프톤','하나금융지주','LG생활건강','HMM','삼성생명','하이브','두산중공업','SK텔레콤','삼성전기','SK바이오사이언스','LG','S-Oil','고려아연','KT&G','우리금융지주','대한항공','삼성에스디에스','현대중공업','엔씨소프트','삼성화재','아모레퍼시픽','KT','포스코케미칼','넷마블','SK아이이테크놀로지','LG이노텍','기업은행']

dfs = []

for name in names:
  try:
    df = get_salary_top(name)
    dfs.append(df)
  except:
    print(f'없음 - {name}')

df_result = pd.concat(dfs)
df_result.sort_values(by='보수',ascending=False)

없음 - 삼성전자우
없음 - 현대차
없음 - 삼성물산
없음 - SK
없음 - 한국전력
없음 - 두산중공업
없음 - KT&G
없음 - 우리금융지주
없음 - 현대중공업
없음 - 삼성화재
없음 - KT
없음 - 포스코케미칼
없음 - 기업은행


,기업명,이름,역할,보수
0,SK하이닉스,박정호,부회장,17486000000
0,SK이노베이션,김준,부회장,13932000000
1,SK하이닉스,박성욱,경영자문위원,12273000000
0,LG,구광모,"대표이사, 회장",8177000000
0,삼성전자,경계현,고문,8036000000
...,...,...,...,...
4,SK아이이테크놀로지,이병인,부사장,567000000
1,HMM,박진기,사내이사,536002000
4,카카오뱅크,허재영,업무집행책임자\n (퇴직임원),528000000
3,KB금융,김상덕,부장,509000000


In [13]:
# // 최대 주주의 주식 변동 모아보기- 종목 코드 가져오기

corp_code = df_listed[df_listed['corp_name'] == '카카오'].iloc[0,0]
corp_code

'00258801'

In [15]:
#  // 주요 주주 지분율 보기

data = dart_fss.api.info.hyslr_sttus('00258801', '2021', '11011')
df = pd.DataFrame(data['list'])

df

,rcept_no,corp_cls,corp_code,corp_name,stock_knd,nm,relate,bsis_posesn_stock_co,bsis_posesn_stock_qota_rt,trmend_posesn_stock_co,trmend_posesn_stock_qota_rt,rm,stlm_dt
0,20240320000386,Y,00258801,카카오,보통주,김범수,본인,"12,500,631",14.12,"59,124,547",13.26,증여 주식분할,2021-12-31
1,20240320000386,Y,00258801,카카오,보통주,김은정,특수관계인,0,0.00,"71,850",0.02,"수증, 주식분할",2021-12-31
2,20240320000386,Y,00258801,카카오,보통주,김진원,특수관계인,634,0.00,0,0.00,"수증, 주식분할, 단순매도",2021-12-31
3,20240320000386,Y,00258801,카카오,보통주,김행자,특수관계인,0,0.00,"100,000",0.02,"수증, 주식분할",2021-12-31
4,20240320000386,Y,00258801,카카오,보통주,김현수,특수관계인,0,0.00,0,0.00,단순 매매,2021-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,20240320000386,Y,00258801,카카오,보통주,황인호,특수관계인,"7,200",0.01,0,0.00,"자사주 상여금 수취, 주식분할, 주식매수선택권 행사,특별관계해소",2021-12-31
75,20240320000386,Y,00258801,카카오,보통주,김예빈,특수관계인,0,0.00,"268,000",0.06,"수증, 주식분할",2021-12-31
76,20240320000386,Y,00258801,카카오,보통주,김유태,특수관계인,0,0.00,"14,700",0.00,"수증, 주식분할",2021-12-31
77,20240320000386,Y,00258801,카카오,보통주,계,NaN,"22,542,030",25.47,"107,844,461",24.19,-,2021-12-31
